# Data Parallel

In [ ]:
import os

import wandb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# CONSTANTS
DOUBLE_WIDTH_PT = 487.8225 
DOUBLE_COL_WIDTH_INCH = DOUBLE_WIDTH_PT / 72.27

SMALL_FONT = 10
TICK_FONT = 8

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": SMALL_FONT,
    "axes.labelsize": SMALL_FONT,
    "axes.titlesize": SMALL_FONT,
    "xtick.labelsize": TICK_FONT,
    "ytick.labelsize": TICK_FONT,
})

In [ ]:
# BUILD CSV FILE

DP_OUTPUT_PATH = "dp_metrics.csv"
NUM_WORKERS_OUTPUT_PATH = "num_workers_metrics.csv"

if os.path.exists(DP_OUTPUT_PATH):
    METRICS = pd.read_csv(DP_OUTPUT_PATH)
else:
    METRICS_ROWS = []

    def extract_dp_metrics(wandb_run):
        # Extract history
        history = wandb_run.history()
        subset = history[(history["_step"] >= 40) & (history["_step"] <= 50)]

        # Extract num_workers from the config
        config = wandb_run.config
        num_workers = config.get("num_workers")

        return {
            "num_workers": num_workers,
            "iteration_time_mean": subset["iteration-time"].mean(),
            "iteration_time_std": subset["iteration-time"].std(),
            "tokens_per_sec_per_gpu_mean": subset["tokens-per-sec-per-gpu"].mean(),
            "tokens_per_sec_per_gpu_std": subset["tokens-per-sec-per-gpu"].std(),
        }

    # Extract DP Sweep Metrics
    SIZE_DP_TO_WANDB_PATH = {
        "125m": {
            1:  "LSAIE/gipfelsturm/q4w6xzxr",
            2:  "LSAIE/gipfelsturm/ouuv1is5",
            4:  "LSAIE/gipfelsturm/hk9kv8tu",
            8:  "LSAIE/gipfelsturm/zgol0lsq",
            16: "LSAIE/gipfelsturm/0zyd3nfd",
        },
        "350m": {
            1:  "LSAIE/gipfelsturm/l9f7lxph",
            2:  "LSAIE/gipfelsturm/yfewdobu",
            4:  "LSAIE/gipfelsturm/f8k55wzl",
            8:  "LSAIE/gipfelsturm/lfmuizoj",
            16: "LSAIE/gipfelsturm/bve6bb3i",
        },
        "760m": {
            1:  "LSAIE/gipfelsturm/aruh6cq9",
            2:  "LSAIE/gipfelsturm/3zb7aw2j",
            4:  "LSAIE/gipfelsturm/dt0isutw",
            8:  "LSAIE/gipfelsturm/n8ctoocb",
            16: "LSAIE/gipfelsturm/5qw3lp2t",
        },
        "1.5b": {
            1:  "LSAIE/gipfelsturm/j1sx6j3n",
            2:  "LSAIE/gipfelsturm/94e7gl15",
            4:  "LSAIE/gipfelsturm/f81p6hnc",
            8:  "LSAIE/gipfelsturm/mwdv358e",
            16: "LSAIE/gipfelsturm/6eoerd74",
        },
        "3b": {
            2:  "LSAIE/gipfelsturm/4ahy8m5a",
            4:  "LSAIE/gipfelsturm/jbyfgx6l",
            8:  "LSAIE/gipfelsturm/lw6hzp8c",
            16: "LSAIE/gipfelsturm/blo01f78",
        },
        "8b": {
            4:  "LSAIE/gipfelsturm/6ehbay78",
            8:  "LSAIE/gipfelsturm/klts2ofm",
            16: "LSAIE/gipfelsturm/2f0dxnkm",
        },
    }
    for model_size in SIZE_DP_TO_WANDB_PATH.keys():
        for dp_size in SIZE_DP_TO_WANDB_PATH[model_size].keys():
            run_path = SIZE_DP_TO_WANDB_PATH[model_size][dp_size]

            if not run_path:
                print(f"No run path for {model_size} {dp_size}")
                continue

            api = wandb.Api()
            run = api.run(run_path)
            metrics = extract_dp_metrics(run)

            METRICS_ROWS.append({
                "model_size": model_size,
                "num_workers": metrics["num_workers"],
                "dp_size": dp_size,
                "iteration_time_mean": metrics["iteration_time_mean"],
                "iteration_time_std": metrics["iteration_time_std"],
                "tokens_per_sec_per_gpu_mean": metrics["tokens_per_sec_per_gpu_mean"],
                "tokens_per_sec_per_gpu_std": metrics["tokens_per_sec_per_gpu_std"],
            })

    METRICS = pd.DataFrame(METRICS_ROWS, columns=[
        "model_size",
        "num_workers",
        "dp_size",
        "iteration_time_mean",
        "iteration_time_std",
        "tokens_per_sec_per_gpu_mean",
        "tokens_per_sec_per_gpu_std"
    ])
    METRICS.to_csv(DP_OUTPUT_PATH, index=False)

if os.path.exists(NUM_WORKERS_OUTPUT_PATH):
    NUM_WORKERS_METRICS = pd.read_csv(NUM_WORKERS_OUTPUT_PATH)
else:
    METRICS_ROWS = []

    # Extract Num Workers Sweep Metrics
    SIZE_NUM_WORKERS_TO_WANDB_PATH = {
        "125m": {
            1:  "LSAIE/gipfelsturm/632t1mxk",
            2:  "LSAIE/gipfelsturm/fjwjksnz",
            4:  "LSAIE/gipfelsturm/0guvxpuj",
            8:  "LSAIE/gipfelsturm/rw469k1w",
            16: "LSAIE/gipfelsturm/yy9gcrff",
            32: "LSAIE/gipfelsturm/vvrpimgf",
        },
    }

    def extract_num_workers_metrics(wandb_run):
        history = wandb_run.history()
        config = wandb_run.config
        num_workers = config.get("num_workers")

        return {
            "num_workers": num_workers,
            "iteration_time": history["iteration-time"].tolist(),
            "tokens_per_sec_per_gpu": history["tokens-per-sec-per-gpu"].tolist(),
        }

    for model_size in SIZE_NUM_WORKERS_TO_WANDB_PATH.keys():
        for num_workers in SIZE_NUM_WORKERS_TO_WANDB_PATH[model_size].keys():
            run_path = SIZE_NUM_WORKERS_TO_WANDB_PATH[model_size][num_workers]

            if not run_path:
                print(f"No run path for {model_size} {num_workers}")
                continue

            api = wandb.Api()
            run = api.run(run_path)
            metrics = extract_num_workers_metrics(run)

            METRICS_ROWS.append({
                "model_size": model_size,
                "num_workers": metrics["num_workers"],
                "iteration_time": metrics["iteration_time"],
                "tokens_per_sec_per_gpu": metrics["tokens_per_sec_per_gpu"],
            })

    NUM_WORKERS_METRICS = pd.DataFrame(METRICS_ROWS, columns=[
        "model_size",
        "num_workers",
        "iteration_time",
        "tokens_per_sec_per_gpu",
    ])
    NUM_WORKERS_METRICS.to_csv(NUM_WORKERS_OUTPUT_PATH, index=False)

In [ ]:
# PLOTTING - DP SWEEP

df = pd.read_csv("dp_metrics.csv").sort_values(["model_size", "dp_size"])
df = df[df["num_workers"] == 32]

fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL_WIDTH_INCH, 2.5))

model_size_order = ["125m", "350m", "760m", "1.5b", "3b", "8b"]

for model_size in model_size_order:
    group = df[df["model_size"] == model_size]
    if group.empty:
        continue
    axes[0].plot(group["dp_size"], group["iteration_time_mean"], marker="o", markersize=3, linewidth=1, label=model_size)
    axes[0].fill_between(
        group["dp_size"],
        group["iteration_time_mean"] - group["iteration_time_std"],
        group["iteration_time_mean"] + group["iteration_time_std"],
        alpha=0.15,
    )
    axes[1].plot(group["dp_size"], group["tokens_per_sec_per_gpu_mean"], marker="o", markersize=3, linewidth=1, label=model_size)
    axes[1].fill_between(
        group["dp_size"],
        group["tokens_per_sec_per_gpu_mean"] - group["tokens_per_sec_per_gpu_std"],
        group["tokens_per_sec_per_gpu_mean"] + group["tokens_per_sec_per_gpu_std"],
        alpha=0.15,
    )

axes[0].set_xlabel("DP size", fontsize=SMALL_FONT)
axes[0].set_ylabel("Iteration time (s)", fontsize=SMALL_FONT)
axes[0].set_xscale("log", base=2)
axes[0].tick_params(axis="both", labelsize=TICK_FONT)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("DP size", fontsize=SMALL_FONT)
axes[1].set_ylabel("Tokens / sec / GPU", fontsize=SMALL_FONT)
axes[1].set_xscale("log", base=2)
axes[1].tick_params(axis="both", labelsize=TICK_FONT)
axes[1].grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    fontsize=TICK_FONT,
    title_fontsize=TICK_FONT,
    loc="upper center",
    ncol=len(model_size_order),
    columnspacing=3.5,
    bbox_to_anchor=(0.5, 1.05),
    framealpha=0.0,
)

plt.tight_layout()
plt.savefig("dp_sweep.pdf", bbox_inches="tight")
plt.show()


In [ ]:
# LATEX TABLE - DP SWEEP

df = pd.read_csv("dp_metrics.csv")

MODEL_ORDER = ["125m", "350m", "760m", "1.5b", "3b", "8b"]
DP_SIZES = [1, 2, 4, 8, 16]
LABELS = {"125m": "125M", "350m": "350M", "760m": "760M", "1.5b": "1.5B", "3b": "3B", "8b": "8B"}

lookup = df.set_index(["model_size", "dp_size"])

def fmt_iter(v):
    return "-" if pd.isna(v) else f"{v:.2f}"

def fmt_tok(v):
    return "-" if pd.isna(v) else f"{int(v)}"

for model in MODEL_ORDER:
    iters, toks = [], []
    for dp in DP_SIZES:
        if (model, dp) in lookup.index:
            row = lookup.loc[(model, dp)]
            iters.append(fmt_iter(row["iteration_time_mean"]))
            toks.append(fmt_tok(row["tokens_per_sec_per_gpu_mean"]))
        else:
            iters.append("-")
            toks.append("-")
    cells = [LABELS[model]] + iters + toks
    print(" & ".join(cells) + r" \\")


In [ ]:
# PLOTTING - NUM WORKERS SWEEP

df = pd.read_csv("num_workers_metrics.csv")
df = df[df["model_size"] == "125m"].sort_values("num_workers")


_SERIES_GLOBALS = {"__builtins__": {}, "nan": np.nan, "inf": np.inf}


def parse_series(value):
    if isinstance(value, str):
        value = eval(value, _SERIES_GLOBALS)
    arr = np.array(value, dtype=float)
    return arr


fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COL_WIDTH_INCH, 2.5))

for _, row in df.iterrows():
    num_workers = int(row["num_workers"])
    iter_time = parse_series(row["iteration_time"])
    tokens = parse_series(row["tokens_per_sec_per_gpu"])

    iter_mask = np.isfinite(iter_time)
    tokens_mask = np.isfinite(tokens)

    axes[0].plot(
        np.arange(len(iter_time))[iter_mask],
        iter_time[iter_mask],
        linewidth=1,
        label=str(num_workers),
    )
    axes[1].plot(
        np.arange(len(tokens))[tokens_mask],
        tokens[tokens_mask],
        linewidth=1,
        label=str(num_workers),
    )

axes[0].set_xlabel("Step", fontsize=SMALL_FONT)
axes[0].set_ylabel("Iteration time (s)", fontsize=SMALL_FONT)
axes[0].tick_params(axis="both", labelsize=TICK_FONT)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Step", fontsize=SMALL_FONT)
axes[1].set_ylabel("Tokens / sec / GPU", fontsize=SMALL_FONT)
axes[1].tick_params(axis="both", labelsize=TICK_FONT)
axes[1].grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    fontsize=TICK_FONT,
    title_fontsize=TICK_FONT,
    loc="upper center",
    ncol=len(labels),
    columnspacing=3.5,
    bbox_to_anchor=(0.5, 1.05),
    framealpha=0.0,
)

plt.tight_layout()
plt.savefig("num_workers_sweep.pdf", bbox_inches="tight")
plt.show()